# Práctica guiada: del modelo legado al artefacto explícito

En la semana 3 usamos un payload **joblib** con tres claves: el estimador, los nombres de las features y la versión del modelo. En esta práctica vamos a separar responsabilidades:

- **model.joblib**: el objeto ejecutable;
- **manifest.json**: el contrato que permite saber si el objeto es compatible;
- código y pruebas: las reglas de validación y el comportamiento ante errores.

## Resultado esperado

Al terminar tendrás un manifiesto mínimo escrito como JSON, una lista de invariantes y una matriz de fallos que otra persona podrá implementar en la segunda clase.

## 1. Inspeccionar el punto de partida

Primero comprueba qué columnas recibe el módulo de inferencia de la semana 3. No cambies todavía el modelo ni ejecutes predicciones.

In [1]:
from pathlib import Path
import csv

week_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "modules/04-model-packaging").is_dir()
)
input_path = week_root / "assets/03-wine-quality/inference_samples.csv"
if not input_path.exists():
    input_path = week_root.parent / "semana3/assets/03-wine-quality/inference_samples.csv"

with input_path.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)
print(f"Fichero: {input_path}")
print(f"Columnas: {reader.fieldnames}")
print(f"Filas de ejemplo: {len(rows)}")

Fichero: semana3/assets/03-wine-quality/inference_samples.csv
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Filas de ejemplo: 5


## 2. Diseñar la frontera del artefacto

Completa esta tabla antes de escribir código. La pregunta clave es: **¿qué necesita un proceso de inferencia para decidir que puede utilizar el artefacto?**

| Elemento | ¿Dónde vive? | ¿Qué riesgo evita? |
| --- | --- | --- |
| Estimador entrenado | **model.joblib** | Es el objeto ejecutable. Separarlo permite revisar los metadatos **sin** deserializarlo, porque `joblib.load` puede ejecutar código. |
| Versión del esquema | **manifest.json** (`schema_version`) | Que el cargador intente leer un bundle de un formato que no conoce (claves con otro significado). |
| Orden y nombres de features | **manifest.json** (`feature_names`) + comparado en el **código** con `FEATURE_NAMES` | Que el modelo reciba las 11 columnas en otro orden y prediga mal **sin dar error**. |
| Versión del preprocesado | **manifest.json** (`preprocessing_version`) + comparada en el **código** con `PREPROCESSING_VERSION` | Usar el modelo con una transformación de datos distinta a la del entrenamiento. |
| Etiquetas de salida permitidas | **manifest.json** (`output_labels`) + validadas en el **código** con `WineQualityPrediction` | Que la respuesta exponga una categoría que la interfaz no conoce (`unknown`). |
| Versión del modelo | **manifest.json** (`model_version`) | Salidas no trazables: no saber qué modelo generó cada predicción. |
| Tipo de estimador | **manifest.json** (`estimator_type`) + comprobado en el **código** contra el binario cargado | Que se haya cambiado el binario por otro objeto distinto del declarado. |
| Reglas de validación (rangos, `extra="forbid"`, todo o nada) | **Código** (`contracts.py`, `artifact.py`, `predict_file.py`) y **tests** | Son comportamiento, no datos: se versionan con el código y se prueban. |


## 3. Práctica 2 — Construir un manifiesto mínimo

Completa el borrador de Python de la siguiente celda y ejecútalo para producir un JSON legible. Mantén los nombres de campo estables: el manifiesto es una interfaz entre quien empaqueta el modelo y quien lo ejecuta.

~~~json
{
  "schema_version": "",
  "model_version": "",
  "preprocessing_version": "",
  "feature_names": [],
  "output_labels": [],
  "estimator_type": ""
}
~~~

Comprueba antes de continuar que:

- aparecen exactamente las seis claves obligatorias;
- `feature_names` tiene once nombres y no incluye `sample_id`;
- `output_labels` representa las categorías que la interfaz podrá devolver.

Decide también qué debe pasar si:

1. falta **manifest.json**;
2. aparecen features desconocidas o en un orden distinto;
3. el modelo devuelve una etiqueta que no está en **output_labels**;
4. una fila del CSV no cumple el contrato de entrada.

**Decisiones:**

1. **Falta `manifest.json`:** no se carga el modelo (`FileNotFoundError`). Sin manifiesto no se puede saber si el binario es compatible.
2. **Features desconocidas o en otro orden:** se rechaza el bundle al validar el manifiesto, antes de deserializar el modelo.
3. **El modelo devuelve una etiqueta fuera de `output_labels`:** la inferencia falla con `ValueError` y no se escribe salida.
4. **Una fila no cumple el contrato de entrada:** `WineQualityRequest` la rechaza, el comando para y no publica CSV.

In [2]:
import json

feature_names = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
]

# No modifiques este orden: procede del contrato de la semana 3.
legacy_model_version = "wine-quality-rf-v1"  # la versión del payload de S3

manifest_draft = {
    "schema_version": "wine-quality-bundle-v1",  # formato del bundle
    "model_version": legacy_model_version,  # qué modelo es
    "preprocessing_version": "wine-red-features-v1",  # qué preprocesado espera
    "feature_names": feature_names,  # las 11, en orden
    "output_labels": ["needs_review", "acceptable", "excellent"],  # contrato S3
    "estimator_type": "RandomForestClassifier",  # clase del estimador entregado
}

required_keys = {
    "schema_version",
    "model_version",
    "preprocessing_version",
    "feature_names",
    "output_labels",
    "estimator_type",
}

assert set(manifest_draft) == required_keys
assert len(manifest_draft["feature_names"]) == 11
assert "sample_id" not in manifest_draft["feature_names"]

print(json.dumps(manifest_draft, indent=2, ensure_ascii=False))
print("\nLa estructura es válida.")

{
  "schema_version": "wine-quality-bundle-v1",
  "model_version": "wine-quality-rf-v1",
  "preprocessing_version": "wine-red-features-v1",
  "feature_names": [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol"
  ],
  "output_labels": [
    "needs_review",
    "acceptable",
    "excellent"
  ],
  "estimator_type": "RandomForestClassifier"
}

La estructura es válida.


## 4. Práctica 3 — Anticipar fallos

Antes de ver la demo del profesorado, completa la tabla. No describas solo el error: indica en qué frontera debería detenerse el flujo.

| Cambio | Frontera que debe rechazarlo | Mensaje útil esperado | ¿Se escribe CSV? |
| --- | --- | --- | --- |
| Se intercambian `density` y `alcohol` | **Carga del bundle**: validación del manifiesto, antes de deserializar `model.joblib` | `feature_names no coincide con FEATURE_NAMES` | **No** |
| Cambia `preprocessing_version` | **Carga del bundle**: validación del manifiesto | `preprocessing_version incompatible: se esperaba wine-red-features-v1` | **No** |
| El modelo devuelve `unknown` | **Inferencia**: validación de la salida (`output_labels` / `WineQualityPrediction`) | `quality_band no permitida: unknown` | **No** (el error para todo el lote) |
| La petición incluye `ph = 99` | **Entrada**: `WineQualityRequest`, antes de preprocesar | `Fila red-001: ph … less than or equal to 4.5` | **No** |
| `estimator_type` no coincide con el binario | **Carga del bundle**: después de deserializar, antes de inferir | `estimator_type incompatible` | **No** |


### Comprobación: romper el contrato a propósito

La celda siguiente aplica cada cambio al manifiesto propuesto y comprueba qué invariante lo detecta. Es una versión mínima, sin Pydantic, de lo que hará `ArtifactManifest` en la clase 2.

In [3]:
EXPECTED = {
    "schema_version": "wine-quality-bundle-v1",
    "preprocessing_version": "wine-red-features-v1",
    "feature_names": feature_names,
    "output_labels": ["needs_review", "acceptable", "excellent"],
}


def check_manifest(manifest, estimator_class_name):
    """Devuelve el primer invariante que falla, o 'OK'."""
    if set(manifest) != required_keys:  # 1. claves exactas
        return "claves incorrectas"
    for key, value in EXPECTED.items():  # 2-5. versiones, orden y etiquetas
        if manifest[key] != value:
            return f"{key} incompatible"
    if manifest["estimator_type"] != estimator_class_name:  # 6. tipo de estimador
        return "estimator_type incompatible"
    return "OK"


swapped = list(feature_names)
i, j = swapped.index("density"), swapped.index("alcohol")
swapped[i], swapped[j] = swapped[j], swapped[i]  # intercambiamos dos columnas

cases = {
    "manifiesto correcto": (manifest_draft, "RandomForestClassifier"),
    "density <-> alcohol": ({**manifest_draft, "feature_names": swapped}, "RandomForestClassifier"),
    "otro preprocesado": ({**manifest_draft, "preprocessing_version": "wine-red-features-v2"}, "RandomForestClassifier"),
    "etiqueta 'unknown'": ({**manifest_draft, "output_labels": ["unknown"]}, "RandomForestClassifier"),
    "otro estimador": (manifest_draft, "DummyClassifier"),
    "clave extra": ({**manifest_draft, "author": "yo"}, "RandomForestClassifier"),
}

for name, (manifest, estimator_name) in cases.items():
    print(f"{name:22} -> {check_manifest(manifest, estimator_name)}")

manifiesto correcto    -> OK
density <-> alcohol    -> feature_names incompatible
otro preprocesado      -> preprocessing_version incompatible
etiqueta 'unknown'     -> output_labels incompatible
otro estimador         -> estimator_type incompatible
clave extra            -> claves incorrectas


## 5. Práctica 4 — Preparar la implementación de la segunda clase

Antes de pasar al taller, deja cerradas estas decisiones:

- ¿Qué campos son obligatorios y cuáles tienen valores por defecto?
- ¿Aceptarás campos desconocidos en el manifiesto?
- ¿Validarás el manifiesto antes de cargar el objeto serializado?
- ¿El CSV de salida se escribe fila a fila o solo después de validar todas las entradas?
- ¿Qué mensaje verá la persona que ejecuta la CLI cuando el contrato falle?
- ¿Qué ocurre si una segunda fila es inválida después de haber procesado una primera fila correcta?

La segunda clase convertirá estas decisiones en **ArtifactManifest**, **save_model_bundle**, **load_model_bundle**, inferencia y pruebas.

**Respuestas:**

- **¿Qué campos son obligatorios y cuáles tienen valores por defecto?** Las **seis** claves son obligatorias y **ninguna** tiene valor por defecto. Un valor por defecto escondería un manifiesto incompleto.
- **¿Aceptarás campos desconocidos en el manifiesto?** **No** (`extra="forbid"`). Una clave desconocida indica otro formato o un error de escritura.
- **¿Validarás el manifiesto antes de cargar el objeto serializado?** **Sí.** Primero se lee y valida el JSON y solo después se ejecuta `joblib.load`. Así no se deserializa un binario incompatible.
- **¿El CSV de salida se escribe fila a fila o después de validar todas las entradas?** **Al final.** Las predicciones se acumulan en memoria y el fichero se escribe solo si todas las filas son válidas: primero en un `.tmp` y después se renombra.
- **¿Qué mensaje verá quien ejecuta la CLI?** Una línea `error: …` por `stderr`, con la fila (`sample_id`) y el campo que falla, y código de salida **2**. No verá un traceback.
- **¿Qué ocurre si la segunda fila es inválida después de una primera correcta?** El comando **para** y **no publica ningún CSV**, ni siquiera con la primera fila. Una salida parcial parece un resultado válido y otro proceso podría consumirla sin saber que le faltan filas.


## Lienzo de la práctica 4.1 (resumen de entrega)

Versión en una página: [LIENZO.md](LIENZO.md).

## 1. Tabla de almacenamiento: binario, manifiesto y código

| Elemento | ¿Dónde vive? | ¿Qué riesgo evita? |
| --- | --- | --- |
| Estimador entrenado | **model.joblib** | Es el objeto ejecutable. Separarlo permite revisar los metadatos **sin** deserializarlo, porque `joblib.load` puede ejecutar código. |
| Versión del esquema | **manifest.json** (`schema_version`) | Que el cargador intente leer un bundle de un formato que no conoce (claves con otro significado). |
| Orden y nombres de features | **manifest.json** (`feature_names`) + comparado en el **código** con `FEATURE_NAMES` | Que el modelo reciba las 11 columnas en otro orden y prediga mal **sin dar error**. |
| Versión del preprocesado | **manifest.json** (`preprocessing_version`) + comparada en el **código** con `PREPROCESSING_VERSION` | Usar el modelo con una transformación de datos distinta a la del entrenamiento. |
| Etiquetas de salida permitidas | **manifest.json** (`output_labels`) + validadas en el **código** con `WineQualityPrediction` | Que la respuesta exponga una categoría que la interfaz no conoce (`unknown`). |
| Versión del modelo | **manifest.json** (`model_version`) | Salidas no trazables: no saber qué modelo generó cada predicción. |
| Tipo de estimador | **manifest.json** (`estimator_type`) + comprobado en el **código** contra el binario cargado | Que se haya cambiado el binario por otro objeto distinto del declarado. |
| Reglas de validación (rangos, `extra="forbid"`, todo o nada) | **Código** (`contracts.py`, `artifact.py`, `predict_file.py`) y **tests** | Son comportamiento, no datos: se versionan con el código y se prueban. |

## 2. Manifiesto propuesto

```json
{
  "schema_version": "wine-quality-bundle-v1",
  "model_version": "wine-quality-rf-v1",
  "preprocessing_version": "wine-red-features-v1",
  "feature_names": ["fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar",
                    "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide", "density",
                    "ph", "sulphates", "alcohol"],
  "output_labels": ["needs_review", "acceptable", "excellent"],
  "estimator_type": "RandomForestClassifier"
}
```

- `model_version` identifica **el modelo entrenado**; `preprocessing_version` identifica **la transformación de los datos**. Son cosas distintas: se puede reentrenar el modelo sin cambiar el preprocesado, y al revés.
- `output_labels` limita las categorías que puede exponer la respuesta a las tres del contrato de la semana 3.

## 3. Invariantes del cargador

Bloquean la carga o la inferencia **antes** de escribir resultados:

1. **El manifiesto existe, es JSON válido y tiene exactamente las 6 claves** (sin claves extra).
2. **`schema_version` y `preprocessing_version` coinciden** con las que entiende el código.
3. **`feature_names` coincide exactamente** (nombres y orden) con `FEATURE_NAMES` de la semana 3.
4. **El estimador tiene `predict` y `predict_proba`**, y su clase coincide con `estimator_type`.
5. **Cada respuesta usa una etiqueta de `output_labels`** y una confianza entre 0 y 1.

## 4. Matriz de compatibilidad (romper el contrato)

| Cambio | Frontera que debe rechazarlo | Mensaje útil esperado | ¿Se escribe CSV? |
| --- | --- | --- | --- |
| Se intercambian `density` y `alcohol` | **Carga del bundle**: validación del manifiesto, antes de deserializar `model.joblib` | `feature_names no coincide con FEATURE_NAMES` | **No** |
| Cambia `preprocessing_version` | **Carga del bundle**: validación del manifiesto | `preprocessing_version incompatible: se esperaba wine-red-features-v1` | **No** |
| El modelo devuelve `unknown` | **Inferencia**: validación de la salida (`output_labels` / `WineQualityPrediction`) | `quality_band no permitida: unknown` | **No** (el error para todo el lote) |
| La petición incluye `ph = 99` | **Entrada**: `WineQualityRequest`, antes de preprocesar | `Fila red-001: ph … less than or equal to 4.5` | **No** |
| `estimator_type` no coincide con el binario | **Carga del bundle**: después de deserializar, antes de inferir | `estimator_type incompatible` | **No** |

## 5. Política de errores: filas inválidas y salida atómica

Si la segunda fila del CSV es inválida después de una primera correcta, el comando **se detiene**, informa de la fila y el campo (`error: Fila red-002: …`), sale con código **2** y **no publica ningún CSV**, ni siquiera con la primera fila.

**Por qué:** una salida parcial parece un resultado válido. Otro proceso podría consumirla sin saber que faltan filas, y el error pasaría desapercibido. Por eso las predicciones se acumulan en memoria y el fichero se escribe solo al final: primero como `.tmp` y después se renombra (`os.replace`), de modo que nunca existe un CSV a medio escribir.

**Límite:** el manifiesto comprueba **compatibilidad**, no seguridad. `joblib.load` puede ejecutar código, así que el binario solo debe cargarse si viene de una fuente de confianza.
